# 04 - Experimental Prioritization

Builds the experimental follow-up artefacts (mutagenesis plan, DSF
plan, ranked summary) for the top N candidates from a screening run.

Use this notebook to pick which screening hits to take to the bench.


## 1. Setup

In [ ]:
import os, subprocess, pathlib
REPO_URL = 'https://github.com/Tommaso-R-Marena/cryptic-ip-pipeline.git'
WORK = pathlib.Path('/content/cryptic-ip-pipeline')
if not WORK.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORK)], check=True)
os.chdir(WORK)
!pip install -q -e . pandas
import pandas as pd, json, pathlib


## 2. Mount Drive (optional, for exporting CSVs)

In [ ]:
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


## 3. Load candidates from a completed screen

In [ ]:
ORGANISM = 'yeast'
screen_csv = pathlib.Path(f'results/screening/{ORGANISM}/screening_results.csv')
if not screen_csv.exists():
    print('No screening output found. Run notebook 02 first.')
    df = pd.DataFrame()
else:
    df = pd.read_csv(screen_csv)
    print('rows:', len(df))
    if 'score' in df.columns:
        display(df.sort_values('score', ascending=False).head(10))


## 4. Generate experimental plans

In [ ]:
!crypticip experimental-plan --organism yeast --top 25

## 5. Inspect mutagenesis plan

In [ ]:
mut_path = pathlib.Path(f'results/experimental/{ORGANISM}/{ORGANISM}_top25_mutagenesis.csv')
if mut_path.exists():
    mut = pd.read_csv(mut_path)
    print('mutagenesis rows:', len(mut))
    display(mut.head(20))
else:
    print('no mutagenesis CSV — did experimental-plan run?')


## 6. Inspect DSF plan

In [ ]:
dsf_path = pathlib.Path(f'results/experimental/{ORGANISM}/{ORGANISM}_top25_dsf.csv')
if dsf_path.exists():
    dsf = pd.read_csv(dsf_path)
    print('DSF rows:', len(dsf))
    display(dsf.head(20))
else:
    print('no DSF CSV — did experimental-plan run?')


## 7. Combined experimental ranking

In [ ]:
plan_path = pathlib.Path(f'results/experimental/{ORGANISM}/{ORGANISM}_top25_experimental_plan.csv')
if plan_path.exists():
    plan = pd.read_csv(plan_path)
    display(plan.head(25))
else:
    print('no experimental_plan CSV')


## 8. Export to Drive (optional)

In [ ]:
if MOUNT_DRIVE:
    import shutil
    target = pathlib.Path('/content/drive/MyDrive/crypticip_experimental')
    target.mkdir(parents=True, exist_ok=True)
    src = pathlib.Path(f'results/experimental/{ORGANISM}')
    if src.exists():
        for p in src.glob('*'):
            shutil.copy(p, target / p.name)
        print('copied to', target)
    else:
        print('no experimental output to copy')
else:
    print('Drive not mounted — toggle MOUNT_DRIVE=True in section 2.')


## 9. Notes

- The mutagenesis plan suggests target residues for charge-reversal /
  alanine substitution based on the basic residues clustered around
  the predicted pocket centre. See `docs/experimental_followup.md`
  for the rationale.
- The DSF plan is a starting point, not a protocol — adapt buffer,
  IP concentration, and ratios to your local SOPs.
- Top-tier hits should also be assessed manually in PyMOL
  (see notebook 02 section 9 for the `.pml` sessions).
